# U-Net Training

## 1. Imports + configuration

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

os.chdir('/content/drive/MyDrive/msc_thesis/notebooks')

In [4]:
!pip install segmentation_models_pytorch

In [5]:
from pathlib import Path
from dataclasses import dataclass
from typing import Sequence, Optional, List, Tuple, Dict
import random

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp

# ----------------------------
# EDIT ONLY THIS CONFIG BLOCK
# ----------------------------
IMG_PATH = Path("../inputs/map/map.tif")
LBL_PATH = Path("../inputs/label/para_three_class.tif")
OUTPUT_DIR = Path("../outputs/unified_unet_mcdo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 3
IN_CHANNELS = 6
NODATA_VALUE = -9999
IGNORE_INDEX = -9999

PATCH_SIZE = 256
BLOCK_SIZE = 2048
SPLIT_RATIOS = (0.70, 0.15, 0.15)

SEED = 42
T_MC = 50
DROP_P = 0.20
EPS = 1e-12

# Training settings inherited from the thesis workflow.
LR = 0.000116
WEIGHT_DECAY = 0.000478
BATCH_SIZE = 4
CLASS_WEIGHT_BETA = 0.205623
AUG_PROB = 0.230711
NOISE_STD = 0.074686
MAX_EPOCHS = 50
PATIENCE = 8

# Set False if a compatible model_best.pt already exists.
RUN_TRAINING = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CKPT_PATH = OUTPUT_DIR / "model_best.pt"
TRAIN_CSV = OUTPUT_DIR / "train_patches.csv"
VAL_CSV = OUTPUT_DIR / "val_patches.csv"
TEST_CSV = OUTPUT_DIR / "test_patches.csv"
MEAN_PATH = OUTPUT_DIR / "train_mean.npy"
STD_PATH = OUTPUT_DIR / "train_std.npy"
HISTORY_PATH = OUTPUT_DIR / "training_history.csv"
RASTER_DIR = OUTPUT_DIR / "metric_rasters"
RASTER_DIR.mkdir(parents=True, exist_ok=True)

def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_global_seed(SEED)

# On Colab, stage Google Drive rasters on the runtime's local disk for faster
# repeated Rasterio window reads. Outputs/checkpoints remain on Google Drive.
# Disable if local runtime storage is insufficient.
STAGE_DRIVE_INPUTS = True

if STAGE_DRIVE_INPUTS and str(IMG_PATH.resolve()).startswith('/content/drive/'):
    import shutil

    local_dir = Path('/content/thesis_inputs')
    local_dir.mkdir(parents=True, exist_ok=True)

    def stage_raster(source: Path) -> Path:
        target = local_dir / source.name
        if not target.exists() or target.stat().st_size != source.stat().st_size:
            print(f"Copying {source.name} to Colab local storage...")
            shutil.copy2(source, target)
        return target

    IMG_PATH = stage_raster(IMG_PATH)
    LBL_PATH = stage_raster(LBL_PATH)

assert IMG_PATH.exists(), f"Missing image raster: {IMG_PATH}"
assert LBL_PATH.exists(), f"Missing label raster: {LBL_PATH}"

print("Device:", DEVICE)
print("Image:", IMG_PATH)
print("Label:", LBL_PATH)
print("Outputs:", OUTPUT_DIR)

Copying map.tif to Colab local storage...
Copying para_three_class.tif to Colab local storage...
Device: cuda
Image: /content/thesis_inputs/map.tif
Label: /content/thesis_inputs/para_three_class.tif
Outputs: ../outputs/unified_unet_mcdo


## 2. Patch utilities, spatial split, normalization, dataset

In [6]:
@dataclass(frozen=True)
class PatchSpec:
    row_off: int
    col_off: int
    height: int
    width: int


def build_training_patch_grid(raster_path: Path, patch_size: int) -> List[PatchSpec]:
    # Training patches are full-size and non-overlapping.
    with rasterio.open(raster_path) as src:
        H, W = src.height, src.width

    return [
        PatchSpec(r, c, patch_size, patch_size)
        for r in range(0, H - patch_size + 1, patch_size)
        for c in range(0, W - patch_size + 1, patch_size)
    ]


def build_full_extent_grid(raster_path: Path, patch_size: int) -> List[PatchSpec]:
    # Inference includes smaller edge windows; dataset pads them before prediction.
    with rasterio.open(raster_path) as src:
        H, W = src.height, src.width

    patches = []
    for r in range(0, H, patch_size):
        h = min(patch_size, H - r)
        for c in range(0, W, patch_size):
            w = min(patch_size, W - c)
            patches.append(PatchSpec(r, c, h, w))
    return patches


def has_valid_label(src, p: PatchSpec) -> bool:
    # Reuse the already-open raster instead of reopening it for every patch.
    y = src.read(1, window=Window(p.col_off, p.row_off, p.width, p.height))
    return bool(np.any((y != IGNORE_INDEX) & (y >= 0) & (y < NUM_CLASSES)))


def spatial_block_split(patches, block_size, ratios=(0.70, 0.15, 0.15), seed=42):
    blocks: Dict[Tuple[int, int], List[PatchSpec]] = {}
    for p in patches:
        key = (p.row_off // block_size, p.col_off // block_size)
        blocks.setdefault(key, []).append(p)

    ids = list(blocks)
    rng = random.Random(seed)
    rng.shuffle(ids)

    n_train = int(ratios[0] * len(ids))
    n_val = int(ratios[1] * len(ids))

    train_ids = set(ids[:n_train])
    val_ids = set(ids[n_train:n_train+n_val])
    test_ids = set(ids[n_train+n_val:])

    train = [p for k in train_ids for p in blocks[k]]
    val = [p for k in val_ids for p in blocks[k]]
    test = [p for k in test_ids for p in blocks[k]]
    return train, val, test


def save_patches(patches, path):
    pd.DataFrame([p.__dict__ for p in patches]).to_csv(path, index=False)


def load_patches(path):
    df = pd.read_csv(path)
    return [PatchSpec(int(r.row_off), int(r.col_off), int(r.height), int(r.width))
            for r in df.itertuples(index=False)]


def estimate_mean_std(img_path, patches, max_patches=400, seed=42):
    rng = np.random.default_rng(seed)
    sample = rng.choice(np.array(patches, dtype=object),
                        size=min(max_patches, len(patches)), replace=False)

    sums = np.zeros(IN_CHANNELS, dtype=np.float64)
    sumsqs = np.zeros(IN_CHANNELS, dtype=np.float64)
    counts = np.zeros(IN_CHANNELS, dtype=np.int64)

    with rasterio.open(img_path) as src:
        for p in sample:
            x = src.read(window=Window(p.col_off, p.row_off, p.width, p.height)).astype(np.float32)
            for c in range(IN_CHANNELS):
                vals = x[c][x[c] != NODATA_VALUE].astype(np.float64)
                if vals.size:
                    sums[c] += vals.sum()
                    sumsqs[c] += np.square(vals).sum()
                    counts[c] += vals.size

    mean = sums / np.maximum(counts, 1)
    var = sumsqs / np.maximum(counts, 1) - mean**2
    return mean.astype(np.float32), np.sqrt(np.maximum(var, 1e-12)).astype(np.float32)


class RasterPatchDataset(Dataset):
    def __init__(self, img_path, lbl_path, patches, mean, std,
                 patch_size=256, augment=False, aug_prob=0.0, noise_std=0.0, seed=42):
        self.img_path = str(img_path)
        self.lbl_path = str(lbl_path)
        self.patches = list(patches)
        self.mean = mean.reshape(-1, 1, 1)
        self.std = std.reshape(-1, 1, 1)
        self.patch_size = patch_size
        self.augment = augment
        self.aug_prob = aug_prob
        self.noise_std = noise_std
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.patches)

    def _augment(self, x, y):
        if self.rng.random() > self.aug_prob:
            return x, y
        if self.rng.random() < 0.5:
            x, y = x[:, :, ::-1].copy(), y[:, ::-1].copy()
        if self.rng.random() < 0.5:
            x, y = x[:, ::-1, :].copy(), y[::-1, :].copy()
        k = int(self.rng.integers(0, 4))
        if k:
            x = np.rot90(x, k, axes=(1, 2)).copy()
            y = np.rot90(y, k, axes=(0, 1)).copy()
        if self.noise_std > 0:
            valid = x != NODATA_VALUE
            noise = self.rng.normal(0, self.noise_std, x.shape).astype(np.float32)
            x = np.where(valid, x + noise, x)
        return x, y

    def __getitem__(self, idx):
        p = self.patches[idx]
        win = Window(p.col_off, p.row_off, p.width, p.height)

        with rasterio.open(self.img_path) as img, rasterio.open(self.lbl_path) as lbl:
            x = img.read(window=win).astype(np.float32)
            y = lbl.read(1, window=win).astype(np.int64)

        if self.augment:
            x, y = self._augment(x, y)

        valid_x = x != NODATA_VALUE
        x = np.where(valid_x, (x - self.mean) / self.std, 0.0).astype(np.float32)

        # Pad edge patches during full-area inference.
        _, h, w = x.shape
        if h != self.patch_size or w != self.patch_size:
            xp = np.zeros((IN_CHANNELS, self.patch_size, self.patch_size), np.float32)
            yp = np.full((self.patch_size, self.patch_size), IGNORE_INDEX, np.int64)
            xp[:, :h, :w] = x
            yp[:h, :w] = y
            x, y = xp, yp

        return torch.from_numpy(x), torch.from_numpy(y), idx


# Build once; reuse existing split/normalization files when available.
if all(p.exists() for p in [TRAIN_CSV, VAL_CSV, TEST_CSV, MEAN_PATH, STD_PATH]):
    train_patches = load_patches(TRAIN_CSV)
    val_patches = load_patches(VAL_CSV)
    test_patches = load_patches(TEST_CSV)
    train_mean, train_std = np.load(MEAN_PATH), np.load(STD_PATH)
    print("Loaded existing splits and normalization.")
else:
    candidate_patches = build_training_patch_grid(LBL_PATH, PATCH_SIZE)
    with rasterio.open(LBL_PATH) as label_src:
        patches = [p for p in candidate_patches if has_valid_label(label_src, p)]
    train_patches, val_patches, test_patches = spatial_block_split(
        patches, BLOCK_SIZE, SPLIT_RATIOS, SEED
    )
    save_patches(train_patches, TRAIN_CSV)
    save_patches(val_patches, VAL_CSV)
    save_patches(test_patches, TEST_CSV)

    train_mean, train_std = estimate_mean_std(IMG_PATH, train_patches, seed=SEED)
    np.save(MEAN_PATH, train_mean)
    np.save(STD_PATH, train_std)
    print("Created splits and training-only normalization.")

print("Train / val / test:", len(train_patches), len(val_patches), len(test_patches))
print("Mean:", train_mean)
print("Std :", train_std)

Loaded existing splits and normalization.
Train / val / test: 1936 408 512
Mean: [ 8356.905   8785.753   8015.2617 18182.428  12833.63    9659.891 ]
Std : [ 897.9387   677.70233  516.11804 2065.2412  2160.6301  1493.2701 ]


## 3. Explicit-dropout ResNet34 U-Net

In [7]:

class ConvBnReluDrop(nn.Module):
    def __init__(self, in_ch, out_ch, p=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p=p),
        )

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, p=0.2):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        self.conv1 = ConvBnReluDrop(in_ch + skip_ch, out_ch, p)
        self.conv2 = ConvBnReluDrop(out_ch, out_ch, p)

    def forward(self, x, skip=None):
        x = self.up(x)

        if skip is not None:
            # Ensure encoder and decoder spatial dimensions match.
            if x.shape[-2:] != skip.shape[-2:]:
                x = nn.functional.interpolate(
                    x,
                    size=skip.shape[-2:],
                    mode="nearest"
                )

            x = torch.cat([x, skip], dim=1)

        return self.conv2(self.conv1(x))


class DropoutDecoder(nn.Module):
    def __init__(
        self,
        encoder_channels,
        decoder_channels=(256, 128, 64, 32, 16),
        p=0.2
    ):
        super().__init__()

        in_ch = encoder_channels[-1]
        skips = list(encoder_channels[-2::-1])

        blocks = []

        for i, out_ch in enumerate(decoder_channels):
            skip_ch = skips[i] if i < len(skips) else 0

            blocks.append(
                DecoderBlock(in_ch, skip_ch, out_ch, p)
            )

            in_ch = out_ch

        self.blocks = nn.ModuleList(blocks)
        self.out_channels = decoder_channels[-1]

    def forward(self, *features):

        # Support both older and newer SMP decoder interfaces.
        if len(features) == 1 and isinstance(
            features[0], (list, tuple)
        ):
            features = features[0]

        x = features[-1]
        skips = list(features[-2::-1])

        for i, block in enumerate(self.blocks):
            skip = skips[i] if i < len(skips) else None
            x = block(x, skip)

        return x


def build_unet_mc_dropout(drop_p=0.2):
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=IN_CHANNELS,
        classes=NUM_CLASSES,
        activation=None,
    )

    decoder = DropoutDecoder(
        model.encoder.out_channels,
        p=drop_p
    )

    model.decoder = decoder

    model.segmentation_head = nn.Conv2d(
        decoder.out_channels,
        NUM_CLASSES,
        kernel_size=1
    )

    return model


model = build_unet_mc_dropout(DROP_P).to(DEVICE)

dropout_count = sum(
    isinstance(m, nn.Dropout2d)
    for m in model.modules()
)

print("Explicit Dropout2d modules:", dropout_count)

assert dropout_count > 0

# Verify the model produces the expected segmentation dimensions.
model.eval()

with torch.no_grad():
    dummy = torch.zeros(
        1,
        IN_CHANNELS,
        PATCH_SIZE,
        PATCH_SIZE,
        device=DEVICE
    )

    output = model(dummy)

assert output.shape == (
    1,
    NUM_CLASSES,
    PATCH_SIZE,
    PATCH_SIZE
), f"Unexpected output shape: {output.shape}"

print("Model output shape:", tuple(output.shape))


Explicit Dropout2d modules: 10
Model output shape: (1, 3, 256, 256)


## 4. Train U-Net (or load existing checkpoint)

In [8]:
def estimate_class_freqs(lbl_path, patches, max_patches=400):
    rng = np.random.default_rng(SEED)
    sample = rng.choice(np.array(patches, dtype=object),
                        size=min(max_patches, len(patches)), replace=False)
    counts = np.zeros(NUM_CLASSES, dtype=np.int64)
    with rasterio.open(lbl_path) as src:
        for p in sample:
            y = src.read(1, window=Window(p.col_off, p.row_off, p.width, p.height))
            valid = (y != IGNORE_INDEX) & (y >= 0) & (y < NUM_CLASSES)
            for c in range(NUM_CLASSES):
                counts[c] += np.sum(y[valid] == c)
    return np.clip(counts / max(counts.sum(), 1), 1e-8, 1.0)


@torch.no_grad()
def macro_iou(model, loader):
    model.eval()
    cm = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.int64)
    for x, y, _ in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x).argmax(1)
        valid = y != IGNORE_INDEX
        idx = y[valid] * NUM_CLASSES + pred[valid]
        cm += torch.bincount(idx.cpu(), minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)

    cm = cm.double()
    tp = cm.diag()
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    iou = tp / (tp + fp + fn + 1e-12)
    present = cm.sum(1) > 0
    return float(iou[present].mean())


if RUN_TRAINING:
    train_ds = RasterPatchDataset(
        IMG_PATH, LBL_PATH, train_patches, train_mean, train_std,
        PATCH_SIZE, augment=True, aug_prob=AUG_PROB, noise_std=NOISE_STD, seed=SEED
    )
    val_ds = RasterPatchDataset(
        IMG_PATH, LBL_PATH, val_patches, train_mean, train_std, PATCH_SIZE
    )

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    freqs = estimate_class_freqs(LBL_PATH, train_patches)
    weights = (1.0 / freqs) ** CLASS_WEIGHT_BETA
    weights /= weights.mean()

    criterion = nn.CrossEntropyLoss(
        ignore_index=IGNORE_INDEX,
        weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE),
    )
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    best_iou = -np.inf
    no_improve = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for x, y, _ in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * x.size(0)

        train_loss = total_loss / len(train_ds)
        val_iou = macro_iou(model, val_loader)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_macro_iou": val_iou})
        print(f"Epoch {epoch:03d} | loss={train_loss:.4f} | val_mIoU={val_iou:.4f}")

        if val_iou > best_iou + 1e-4:
            best_iou = val_iou
            no_improve = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "drop_p": DROP_P,
                "train_mean": train_mean,
                "train_std": train_std,
                "best_val_miou": best_iou,
            }, CKPT_PATH)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print("Early stopping.")
                break

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"
ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
model = build_unet_mc_dropout(float(ckpt.get("drop_p", DROP_P))).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.eval()

print("Loaded best checkpoint:", CKPT_PATH)

Epoch 001 | loss=0.7840 | val_mIoU=0.6374
Epoch 002 | loss=0.5551 | val_mIoU=0.7062
Epoch 003 | loss=0.4431 | val_mIoU=0.7828
Epoch 004 | loss=0.3752 | val_mIoU=0.8010
Epoch 005 | loss=0.3340 | val_mIoU=0.8083
Epoch 006 | loss=0.3118 | val_mIoU=0.8139
Epoch 007 | loss=0.2916 | val_mIoU=0.8047
Epoch 008 | loss=0.2711 | val_mIoU=0.7994
Epoch 009 | loss=0.2526 | val_mIoU=0.8188
Epoch 010 | loss=0.2449 | val_mIoU=0.8093
Epoch 011 | loss=0.2465 | val_mIoU=0.8259
Epoch 012 | loss=0.2326 | val_mIoU=0.7928
Epoch 013 | loss=0.2210 | val_mIoU=0.8302
Epoch 014 | loss=0.2188 | val_mIoU=0.8298
Epoch 015 | loss=0.2246 | val_mIoU=0.8136
Epoch 016 | loss=0.2185 | val_mIoU=0.8298
Epoch 017 | loss=0.2152 | val_mIoU=0.8057
Epoch 018 | loss=0.2076 | val_mIoU=0.8039
Epoch 019 | loss=0.2009 | val_mIoU=0.7270
Epoch 020 | loss=0.2034 | val_mIoU=0.8367
Epoch 021 | loss=0.2011 | val_mIoU=0.8221
Epoch 022 | loss=0.2026 | val_mIoU=0.8416
Epoch 023 | loss=0.1995 | val_mIoU=0.8202
Epoch 024 | loss=0.1969 | val_mIoU